ThinCurr Python Example: Compute time-domain with mode driver in a torus {#doc_tCurr_torus_td}
==========
In this example we demonstrate how to perform a time-domain simulation for a model driven by the plasma mode computed in \ref doc_tCurr_torus_mode.

**Note:** Running this example requires the [h5py](https://www.h5py.org/) and [pyvista](https://pyvista.org/) python packages, which are installable using `pip` or other standard methods.

In [1]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    on_google_colab = True
except:
    import sys
    import os
    tokamaker_python_path = os.getenv('OFT_ROOTPATH')
    if tokamaker_python_path is not None:
        sys.path.append(os.path.join(tokamaker_python_path,'python'))
    on_google_colab = False
    pass
else:
    !pip install -q pyvista wurlitzer ipympl openfusiontoolkit
    print("Google collab detected:")
    %load_ext wurlitzer
%matplotlib inline
%config InlineBackend.figure_format = "retina"

In [2]:
import json
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import pyvista
pyvista.set_jupyter_backend('static') # Comment to enable interactive PyVista plots
plt.rcParams['figure.figsize']=(6,6)
plt.rcParams['font.weight']='bold'
plt.rcParams['axes.labelweight']='bold'
plt.rcParams['lines.linewidth']=2
plt.rcParams['lines.markeredgewidth']=2
%matplotlib inline
%config InlineBackend.figure_format = "retina"

In [3]:
from OpenFUSIONToolkit import OFT_env
from OpenFUSIONToolkit.ThinCurr import ThinCurr
from OpenFUSIONToolkit.ThinCurr.sensor import Mirnov, save_sensors
from OpenFUSIONToolkit.ThinCurr.coils import ThinCurr_Icoil, ThinCurr_XML
from OpenFUSIONToolkit.io import histfile, write_oft_xml

In [ ]:
if on_google_colab:
    myOFT = OFT_env(nthreads=1)
else:
    myOFT = OFT_env(nthreads=4)

In [4]:
xml = ThinCurr_XML()

with open('SPARC_public_simple.json','r') as fid:
    SPARC_geom = json.load(fid)

with open('EQ_currents.json','r') as fid:
    EQ_currents = json.load(fid)

# Set resistivities
xml.set_eta([
    SPARC_geom['vv']['inner']['eta']/SPARC_geom['vv']['inner']['thickness'],
    SPARC_geom['vv']['outer']['eta']/SPARC_geom['vv']['outer']['thickness']
])

# Create PF coils
coil_filament_currents = []
for name, sub_coils in SPARC_geom['coils'].items():
    coil_tmp = ThinCurr_Icoil(name=name)
    for sub_coil in sub_coils:
        coil_tmp.add_subcoil(RZ=np.r_[sub_coil[0], sub_coil[1]])
    xml.add_Icoil(coil_tmp)
    coil_filament_currents.append(EQ_currents[name])

# Create Plasma coils
EQ_filaments = np.loadtxt('EQ_filaments.dat')
for i in range(EQ_filaments.shape[0]):
    coil_tmp = ThinCurr_Icoil(name='Plasma_{0}'.format(i+1))
    coil_tmp.add_subcoil(RZ=np.r_[EQ_filaments[i,0], EQ_filaments[i,1]])
    xml.add_Icoil(coil_tmp)

# Write XML file
write_oft_xml([xml], "SPARC_ThinCurr.xml")

XML file created at SPARC_ThinCurr.xml


## Compute frequency response

### Setup ThinCurr model
We now create a \ref OpenFUSIONToolkit.OFT_env "OFT_env" instance for execution using four threads and a \ref OpenFUSIONToolkit.ThinCurr.ThinCurr "ThinCurr" instance that utilizes that execution environment. Once created, we setup the model from an existing HDF5 and XML mesh definition using \ref OpenFUSIONToolkit.ThinCurr.ThinCurr.setup_model "setup_model()".

We also initialize I/O for this model using \ref OpenFUSIONToolkit.ThinCurr.ThinCurr.setup_io "setup_io()" to enable output of plotting files for 3D visualization in [VisIt](https://visit-dav.github.io/visit-website/index.html), [Paraview](https://www.paraview.org/), or using [pyvista](https://pyvista.org/) below.

In [5]:
tw_SPARC = ThinCurr(myOFT)
tw_SPARC.setup_model(mesh_file='SPARC_public-ThinCurr_approx-homology.h5',xml_filename='SPARC_ThinCurr.xml')
tw_SPARC.setup_io()

#----------------------------------------------
             ____  ____________
            / __ \/ ____/_  __/
           / / / / /_    / /
          / /_/ / __/   / /
          \____/_/     /_/

Base release:        v26.6
Development branch:  main
Revision id:         3db6ce30

Parallelization Info:
  Not compiled with MPI
  # of OpenMP threads =    6

Linear Algebra backend: native

#----------------------------------------------


Creating thin-wall model
  No V(t) driver coils found
  Loading I(t) driver coils
    Masked      0 coils from sensors
  Building holes

  Loading region surface resistivity:
     1  2.6667E-05
     2  2.0000E-05

  Setup complete:
    # of points    =        22456
    # of edges     =        66252
    # of cells     =        43760
    # of holes     =           38
    # of closures  =            0
    # of Vcoils    =            0
    # of Icoils    =          694


### Compute inductance and resistivity matrices
With the model setup, we can now compute the self-inductance matrix using HODLR. When HODLR is used the result is a pointer to the Fortran operator, which is stored at \ref OpenFUSIONToolkit.ThinCurr.ThinCurr.Lmat_hodlr "tw_torus.Lmat_hodlr". As in any other case, by default, the resistivity matrix is not moved to python as it is sparse and converting to dense representation would require an increase in memory. These matrices correspond to the $\textrm{L}$ and $\textrm{R}$ matrices for the physical system

$\textrm{L} \frac{\partial I}{\partial t} + \textrm{R} I = V$

**Note:** Even though HODLR should significantly accelerate the construction of the self-inductance matrix (see \ref doc_thincurr_ex4 for more information) this process may still take some time to complete.

**Note:** The non-zero savings achieved by HODLR compression is reported after the operator is built. Where in this case only 6% of the original memory is needed resulting in a reduction from > 3 GB to ~ 230 MB (over 10x smaller)!

In [6]:
if on_google_colab:
    Mc = tw_SPARC.compute_Mcoil()
    tw_SPARC.compute_Lmat(use_hodlr=True)
    tw_SPARC.compute_Bmat()
else:
    Mc = tw_SPARC.compute_Mcoil(cache_file='Mcoil.save')
    tw_SPARC.compute_Lmat(use_hodlr=True,cache_file='HOLDR_L.save')
    tw_SPARC.compute_Bmat(cache_file='HOLDR_B.save')
tw_SPARC.compute_Rmat()

Reading coil mutual matrices
 Partitioning grid for block low rank compressed operators
   nBlocks =                  32
   Avg block size =          663
   # of SVD =                230
   # of ACA =                209

 Building block low rank inductance operator
   Building hole and Vcoil columns
   Reading HODLR matrix from file: HOLDR_L.save
   Building diagonal blocks
     10%
     20%
     30%
     40%
     50%
     60%
     70%
     80%
     90%
   Building off-diagonal blocks using ACA+
     10%
     20%
     30%
     40%
     50%
     60%
     70%
     80%
     90%
     Compression ratio:   9.4%  ( 4.23E+07/ 4.51E+08)
     Time =  7s          
 Building block low rank magnetic field operator
   Building hole and Vcoil columns
   Reading HODLR matrix from file: HOLDR_B.save
   Building diagonal blocks
     10%
     20%
     30%
     40%
     50%
     60%
     70%
     80%
     90%
   Building off-diagonal blocks using ACA+
     10%
     20%
     30%
     40%
     50%
     60%


## Run time-domain simulation
With the model fully defined we can now use \ref OpenFUSIONToolkit.ThinCurr.ThinCurr.run_td "run_td()" to perform a time-domain simulation. In this case we simulate 80 ms using a timestep of 0.2 ms (400 steps). We also specify using a direct solver for the time-advance (`direct=True`) and set the current in the single I-coil defined in the XML input file as a function of time (`coil_currs`), where the first column specifies time points in ascending order and the remaining columns specify coil currents at each time point.

In [ ]:
CQ_time = 3.E-3

dt = CQ_time/20.0
nsteps = 470

coil_currs = np.zeros((nsteps//4,tw_SPARC.n_icoils+1))
coil_currs[:,0] = np.linspace(0.0,dt*(nsteps-1),nsteps//4)
for i, coil_current in enumerate(coil_filament_currents):
    coil_currs[:,i+1] = coil_current

CQ_ramp = 1.0-np.minimum(coil_currs[:,0]/CQ_time,1.0)
for i, filament_current in enumerate(EQ_filaments[:,2]):
    coil_currs[:,i+len(coil_filament_currents)+1] = filament_current*CQ_ramp
    
tw_SPARC.run_td(dt,nsteps,status_freq=10,plot_freq=1,coil_currs=coil_currs,lin_tol=1.E-8,lin_rtol=1.E-5)


Starting time-domain simulation
  timestep    time           sol_norm     nits    solver time
      10    1.500000E-03    1.2542E+02     105        0.53
      20    3.000000E-03    2.3080E+02     103        0.52
      30    4.500000E-03    2.0967E+02      98        0.50
      40    6.000000E-03    1.9677E+02      96        0.49
      50    7.500000E-03    1.8980E+02      95        0.49
      60    9.000000E-03    1.8533E+02      91        0.50
      70    1.050000E-02    1.8196E+02      89        0.50
      80    1.200000E-02    1.7910E+02      87        0.45
      90    1.350000E-02    1.7648E+02      88        0.47
     100    1.500000E-02    1.7398E+02      86        0.45
     110    1.650000E-02    1.7154E+02      84        0.47
     120    1.800000E-02    1.6914E+02      85        0.44
     130    1.950000E-02    1.6677E+02      86        0.45
     140    2.100000E-02    1.6441E+02      85        0.44
     150    2.250000E-02    1.6209E+02      85        0.51
     160    2.400000

In [ ]:
tw_SPARC.plot_td(nsteps,compute_B=True,plot_freq=1)
plot_data = tw_SPARC.build_XDMF()
ThinCurr_mesh = plot_data['ThinCurr']['smesh']

In [ ]:
icoil_grid = plot_data['ThinCurr']['icoils'].get_pyvista_grid()
grid = ThinCurr_mesh.get_pyvista_grid()
J = ThinCurr_mesh.get_field('J_v',35.E-3)

# Plot mesh and current density using PyVista
p = pyvista.Plotter()
grid["vectors"] = J
grid.set_active_vectors("vectors")
scale = 0.2/(np.linalg.norm(J,axis=1)).max()
arrows = grid.glyph(scale="vectors", orient="vectors", factor=scale)
p.add_mesh(arrows, cmap="turbo", scalar_bar_args={'title': "|J|", "vertical": True, "position_y":0.25, "position_x": 0.0})
p.show()

In [ ]:
J = ThinCurr_mesh.get_field('J_v',35.E-3)
B = ThinCurr_mesh.get_field('B_v',35.E-3)
P = np.cross(J,B,axis=1)

# Plot mesh and current density using PyVista
p = pyvista.Plotter()
grid["vectors"] = P
grid.set_active_vectors("vectors")
scale = 0.2/(np.linalg.norm(P,axis=1)).max()
arrows = grid.glyph(scale="vectors", orient="vectors", factor=scale)
p.add_mesh(arrows, cmap="turbo", scalar_bar_args={'title': "|F/m^2|", "vertical": True, "position_y":0.25, "position_x": 0.0})
p.show()

## Compute net force on cylinder
We now demonstrate how to get the net force on the cylinder by integrating over the surface. To do this we first need some additional information, including the area of each cell (triangle) and the radial unit vector, which we will use to extract a hoop "force". For ThinCurr's triangular grid these can be readily computed from the information in the plotting mesh.

In [ ]:
area = np.zeros((ThinCurr_mesh.nc,))
rhat = np.zeros((ThinCurr_mesh.nc,3))
for i in range(ThinCurr_mesh.nc):
    v1 = ThinCurr_mesh.r[ThinCurr_mesh.lc[i,1],:]-ThinCurr_mesh.r[ThinCurr_mesh.lc[i,0],:]
    v2 = ThinCurr_mesh.r[ThinCurr_mesh.lc[i,2],:]-ThinCurr_mesh.r[ThinCurr_mesh.lc[i,0],:]
    area[i] = np.linalg.norm(np.cross(v1,v2))/2.0
    rcc = (ThinCurr_mesh.r[ThinCurr_mesh.lc[i,2],:]+ThinCurr_mesh.r[ThinCurr_mesh.lc[i,1],:]+ThinCurr_mesh.r[ThinCurr_mesh.lc[i,0],:])/3.0
    rhat[i,:2] = rcc[:2]/np.linalg.norm(rcc[:2])

Then we define a helper function, which computes the desired net force on the cylinder from a cell-centered current density and vertex-centered magnetic field, which we will retrieve from the plot files. The function also supports an optional cell mask, which can be used to restrict the force calculation to specific areas (eg. one region or another in the mesh).

In [ ]:
def compute_force(J_cc,B_v,cell_mask=None):
    B_cc = (B_v[ThinCurr_mesh.lc[:,0],:]+B_v[ThinCurr_mesh.lc[:,1],:]+B_v[ThinCurr_mesh.lc[:,2],:])/3.0
    B_cc = B_cc*area[:,None] # Scale by area for faster integration
    if cell_mask is None:
        F_cc = np.cross(J_cc,B_cc,axis=1)
        Fh = np.sum(F_cc*rhat,axis=None)
    else:
        F_cc = np.cross(J_cc[cell_mask,:],B_cc[cell_mask,:],axis=1)
        Fh = np.sum(F_cc*rhat[cell_mask,:],axis=None)
    return np.r_[np.sum(F_cc,axis=0), Fh]

With this information and functionality we can now loop over the desired time points, computing the force at each time. The results show that a significant inward radial force and a downward vertical force, consistent with the physical arrangement of this test where the current in the coil ramps up in time acting to "crush" the cylinder and push it downward due to the vertical offset. Note that the net forces in the azimuthal plane (XY) are zero, so the hoop "force" is not a realy force per-se but instead a representation of the hoop stress that would be present.

In [ ]:
times = np.linspace(0.0,dt*nsteps,nsteps+1,endpoint=False)
forces_inner = np.zeros((times.shape[0], 4))
forces_outer = np.zeros((times.shape[0], 4))
for i, time in enumerate(times):
    J_cc = ThinCurr_mesh.get_field('J',time)
    B_v = ThinCurr_mesh.get_field('B_v',time)
    forces_inner[i,:] = compute_force(J_cc,B_v,cell_mask=ThinCurr_mesh.reg==1)
    forces_outer[i,:] = compute_force(J_cc,B_v,cell_mask=ThinCurr_mesh.reg==2)


# Plot Ip ramp
fig, ax = plt.subplots(1,1,figsize=(6,5),constrained_layout=True)
ax2 = ax.twinx()
Ip_times = coil_currs[:,0]
Ip = coil_currs[:,-EQ_filaments.shape[0]:].sum(axis=1)
ax2.plot(Ip_times*1.E3,Ip/1.E6,label=r'$I_P$',color='0.5')

# Plot forces
ax.plot(times*1.E3,forces_inner[:,3]/1.E6, label=r'Inner VV')
ax.plot(times*1.E3,forces_outer[:,3]/1.E6, label=r'Outer VV')

# Format plot
ax.grid(True)
ax.set_ylim(-40,20)
ax2.set_ylim(0,12)
ax.legend(loc='upper center',ncol=3)
ax.set_ylabel(r'Inward Hoop Force [MN]')
ax2.set_ylabel(r'$I_p$ [MA]',color='0.5')
ax2.tick_params(color='green', labelcolor='0.5')
ax2.spines['right'].set_edgecolor('0.5')
_ = ax.set_xlabel(r'Time [ms]')